In [1]:
import netket as nk
import jax
import jax.numpy as jnp
from flax.core import FrozenDict
from pennylane_quantum_ansatz import QuantumCircuitAnsatz

# 1) Hilbert & Hamiltonian
L = 6
g = nk.graph.Chain(L)
hi = nk.hilbert.Spin(s=0.5, N=L)
ha = nk.operator.Heisenberg(hi, graph=g)  # H = sum SxSx+SySy+SzSz

# 2) Sampler（PQC 不是自回归，故用 Metropolis）
sa = nk.sampler.MetropolisLocal(hi)

# 3) 模型：我们的 JAX/Flax Ansatz
model = QuantumCircuitAnsatz(n_qubits=L, n_layers=2, entanglement="linear", coef_scale=1.0)

# 4) 初始化参数
key = jax.random.PRNGKey(42)
# NetKet 的模型需要一个形如 (batch, N) 的样本来 init
s0 = jnp.zeros((2, L), dtype=jnp.int32)  # 两个样本
params = model.init(key, s0)  # FrozenDict

# 5) 变分态
vstate = nk.vqs.MCState(sa, model, n_samples=2048)

# 6) 优化器
opt = nk.optimizer.Sgd(learning_rate=5e-3)

# 7) VMC
gs = nk.driver.VMC(ha, opt, variational_state=vstate)
gs.run(n_iter=200, out="log_pqc_heisenberg")


/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 200/200 [00:57<00:00,  3.46it/s, Energy=-6.16-0.00j ± 0.10 [σ²=23.35, R̂=1.0031]]


(JsonLog('log_pqc_heisenberg', mode=write, autoflush_cost=0.005)
   Runtime cost:
   	Log:    0.1000509262084961
   	Params: 0.0734412670135498,)

In [3]:
from pyscf import gto, scf, fci
import netket as nk; import netket.experimental as nkx

bond_length = 1.5109
geometry = [('Li', (0., 0., -bond_length/2)), ('H', (0., 0., bond_length/2))]
mol = gto.M(atom=geometry, basis='STO-3G')

mf = scf.RHF(mol).run(verbose=0)
E_hf = sum(mf.scf_summary.values())

E_fci = fci.FCI(mf).kernel()[0]

ha = nkx.operator.from_pyscf_molecule(mol)
ha
# E0 = -7.88253, E_fci = -7.88253

ParticleNumberAndSpinConservingFermioperator2nd(_hilbert=SpinOrbitalFermions(n_orbitals=6, s=1/2, n_fermions=4, n_fermions_per_spin=(2, 2)), _operator_data={'diag': {(0, ()): (Array(0, dtype=int32), Array([], shape=(1, 1, 0), dtype=int32), Array([1.0507192], dtype=float64)), (2, (0, 1)): (None, None, COOArray(_coords=Array([[0],
       [1],
       [2],
       [3],
       [4],
       [5]], dtype=int64), data=Array([-4.74671784, -1.52769814, -1.13174662, -1.14428486, -1.14428486,
       -0.93546292], dtype=float64), shape=(6,), fill_value=0)), (4, (0, 1)): (None, None, COOArray(_coords=Array([[1, 0],
       [2, 0],
       [2, 1],
       [3, 0],
       [3, 1],
       [3, 2],
       [4, 0],
       [4, 1],
       [4, 2],
       [4, 3],
       [5, 0],
       [5, 1],
       [5, 2],
       [5, 3],
       [5, 4]], dtype=int64), data=Array([-0.36348199, -0.37440379, -0.21404788, -0.38649058, -0.25067256,
       -0.24088564, -0.38649058, -0.25067256, -0.24088564, -0.26233809,
       -0.35431155, 